In [2]:
import msprime
import numpy as np
import pandas as pd
import math
from collections import defaultdict

In [3]:
dem = msprime.Demography()
dem.add_population(name="A", initial_size=4000)
ts = msprime.sim_ancestry(samples={"A": 20}, demography=dem, sequence_length=3e7, recombination_rate=1e-7, random_seed=1, model = 'smc') 
mutated_ts = msprime.sim_mutations(ts, rate=1e-7, random_seed=1)


In [4]:
mutated_ts.num_trees

201598

In [5]:
def all_ibd_segments(ts):
    """
    Manually extracts IBD segments by iterating trees, 
    but MERGES adjacent trees if the TMRCA for the pair is unchanged.
    """
    # Initialize a matrix to store current segment starts for every pair
    n = ts.num_samples
    current_mrca = np.zeros((n, n)) - 1
    current_start = np.zeros((n, n))
    
    # Store completed segments: segments[u][v] = [len1, len2, ...]
    segments = defaultdict(lambda: defaultdict(list))
    
    # Iterate continuously through the genome
    for tree in ts.trees():
        interval_end = tree.interval.right
        
        # This is efficient for small n, slow for large n
        # For every pair, check their MRCA time/node
        for i in range(n):
            for j in range(i + 1, n):
                mrca_node = tree.mrca(i, j)
                
                # If this is the first tree, initialize
                if current_mrca[i, j] == -1:
                    current_mrca[i, j] = mrca_node
                    current_start[i, j] = tree.interval.left
                
                # If MRCA changed, the segment ended. Record it.
                elif current_mrca[i, j] != mrca_node:
                    # Calculate length (fraction of genome or bp)
                    # Here we store fraction to match your 'l' definition
                    seg_len = (tree.interval.left - current_start[i, j]) / ts.sequence_length
                    segments[i][j].append(seg_len)
                    
                    # Start new segment
                    current_mrca[i, j] = mrca_node
                    current_start[i, j] = tree.interval.left
                    
    # Flush the final segments at the end of the chromosome
    for i in range(n):
        for j in range(i + 1, n):
            seg_len = (ts.sequence_length - current_start[i, j]) / ts.sequence_length
            segments[i][j].append(seg_len)
            
    return segments

In [6]:
M = all_ibd_segments(mutated_ts)

In [7]:
len(M)

39

In [66]:
bins = [[0.8,1],[1,300]]
total_frac = {bin[0]: 0 for bin in bins}
for bin in bins:
    for i in range(40):
        for j in range(i+1,40):
            seg = M[i][j]
            total_frac[bin[0]] += sum([s for s in seg if bin[0] < s * 300 <= bin[1]])



In [67]:
total_frac[bins[0][0]]/(20*39)

np.float64(0.003021529487179488)

In [68]:
total_frac[bins[1][0]]/(20*39)

np.float64(0.01338418931623931)

In [61]:
def calculate_ibd_fractions(ts, bins, cm_per_unit=1e-5, num_bootstraps=1000):    

    
    sample_nodes = ts.samples()
    node_to_pop = ts.nodes_population[sample_nodes]
    pop_ids = np.unique(node_to_pop)
    num_pops = len(pop_ids)
    

    pop_samples = defaultdict(list)
    for u in sample_nodes:
        pop_samples[node_to_pop[u]].append(u)

    results = {b_i: defaultdict(lambda: defaultdict(float)) 
               for b_i in range(len(bins))}
    
    genome_length = ts.sequence_length * cm_per_unit
    
    # Filter tiny segments
    min_bin_val = min(b[0] for b in bins)
    min_span_ts_units = min_bin_val / cm_per_unit
    
    ibd_iter = ts.ibd_segments(
        store_pairs=True, 
        store_segments=True,
        min_span=min_span_ts_units
    )

    print(f"Iterating IBD segments (min_span={min_span_ts_units:.2f})...")
    
    for (u, v), segments in ibd_iter.items(): 
        p_u = ts.nodes_population[u]
        p_v = ts.nodes_population[v]
        p_i, p_j = sorted((p_u, p_v))
        
        # Unique identifier for this specific pair of individuals
        pair_key = tuple(sorted((u, v)))
        
        for seg in segments:
            seg_len = (seg.right - seg.left) * cm_per_unit
            
            for b_i, (min_len, max_len) in enumerate(bins):
                if min_len < seg_len <= max_len:
                    # FIX 3: Store fraction for the pair using tuple key (p_i, p_j)
                    # This matches how the bootstrap loop tries to retrieve it later.
                    results[b_i][(p_i, p_j)][pair_key] += (seg_len/genome_length)
                    break 
    
    final_mean_matrix = {}
    final_var_matrix = {}

    for b_i in results:
        mean_matrix = np.zeros((num_pops, num_pops))
        var_matrix = np.zeros((num_pops, num_pops))

        for i in range(num_pops):
            for j in range(i, num_pops):
                # Calculate total theoretical pairs (N)
                # Now this works because pop_samples[i] is a list
                if i == j:
                    n = len(pop_samples[i]) #
                    num_pairs = n * (n - 1) // 2
                else:
                    num_pairs = len(pop_samples[i]) * len(pop_samples[j])
                
                if num_pairs == 0:
                    continue

                # Retrieve observed non-zero fractions
                # This works now because we stored data with key (i, j)
                observed_dict = results[b_i].get((i, j), {})
                observed_values = np.array(list(observed_dict.values()))
                
                # The rest are zeros
                count_zeros = num_pairs - len(observed_values)
                
                # Construct the full population of pairs
                full_population = np.concatenate([
                    observed_values, 
                    np.zeros(count_zeros)
                ])

                # A. Original Mean
                original_mean = np.mean(full_population)
                
                # B. Bootstrap Variance
                if num_bootstraps > 0:
                    boot_samples = np.random.choice(full_population, size=(num_bootstraps, num_pairs), replace=True)
                    boot_means = np.mean(boot_samples, axis=1)
                    boot_var = np.var(boot_means)
                else:
                    boot_var = 0.0

                # Fill Matrices
                mean_matrix[i, j] = mean_matrix[j, i] = original_mean
                var_matrix[i, j] = var_matrix[j, i] = boot_var

        final_mean_matrix[b_i] = mean_matrix
        final_var_matrix[b_i] = var_matrix

    return final_mean_matrix, final_var_matrix

In [62]:
a,b = calculate_ibd_fractions(mutated_ts, bins)

Iterating IBD segments (min_span=80000.00)...


In [64]:
(a[0] + a[1])

array([[0.01209684]])

In [79]:
a[0]

array([[0.00169304]])

In [63]:
print(mutated_ts.ibd_segments(min_span=0.8/1e-5))

╔══════════════════════════╗
║IdentitySegments          ║
╠══════════════╤═══════════╣
║Parameters:   │           ║
║max_time      │        inf║
║min_span      │    80000.0║
║store_pairs   │      False║
║store_segments│      False║
║Results:      │           ║
║num_segments  │       1373║
║total_span    │283065957.0║
╚══════════════╧═══════════╝



In [58]:
283065957.0/(20*39)/(3e7)

0.01209683576923077

In [69]:
def fraction(u,v,N):
    return (1+8 * N * u)/(1 + 4 * N * u)**2 - (1+8 * N * v)/(1 + 4 * N * v)**2

In [76]:
def frac(u,N):
    return (4 * N * u)/(2 * N * u + 1)**2

In [90]:
frac(0.008,8000) - frac(0.01,8000)

0.003038488152585195

In [80]:
def calculate_ibd_fractions_mrca(ts, bins, cm_per_unit=1e-6, num_bootstraps=1000):
    """
    Calculates IBD fractions using the strict 'MRCA-span' definition.
    Segments are defined by continuous TMRCA, ignoring changes in topology 
    that do not alter the most recent common ancestor.
    """
    
    # 1. Setup Population Mappings
    sample_nodes = ts.samples()
    num_samples = len(sample_nodes)
    node_to_pop = ts.nodes_population[sample_nodes]
    pop_ids = np.unique(node_to_pop)
    num_pops = len(pop_ids)

    pop_samples = defaultdict(list)
    for u in sample_nodes:
        pop_samples[node_to_pop[u]].append(u)

    # 2. Initialize Results Container
    # structure: results[bin_index][(pop_i, pop_j)][pair_key] = total_fraction
    results = {b_i: defaultdict(lambda: defaultdict(float)) 
               for b_i in range(len(bins))}
    
    genome_length_cm = ts.sequence_length * cm_per_unit

    # 3. Iterate Trees to Extract MRCA-span Segments
    # We maintain the state of the current segment for every pair of samples.
    # state[(u, v)] = (current_mrca_node, start_coordinate_bp)
    current_state = {}
    
    # Initialize with the first tree
    tree_iter = ts.trees()
    first_tree = next(tree_iter)
    
    # Pre-calculate pairs to iterate (u, v)
    # Using indices to access the sample_nodes array is faster
    pairs = []
    for i in range(num_samples):
        for j in range(i + 1, num_samples):
            u, v = sample_nodes[i], sample_nodes[j]
            pairs.append((u, v))
            
            # Initialize state
            mrca = first_tree.mrca(u, v)
            current_state[(u, v)] = (mrca, first_tree.interval.left)

    print(f"Scanning trees for MRCA segments ({len(pairs)} pairs)...")

    # Helper to process a finished segment
    def process_segment(u, v, length_bp):
        length_cm = length_bp * cm_per_unit
        
        # Binning logic
        for b_i, (min_len, max_len) in enumerate(bins):
            # Note: Strict inequality matching your request (min < len <= max)
            # Adjust if you need inclusive lower bound
            if min_len < length_cm <= max_len:
                p_u = node_to_pop[u]
                p_v = node_to_pop[v]
                p_i, p_j = sorted((p_u, p_v))
                pair_key = tuple(sorted((u, v)))
                
                results[b_i][(p_i, p_j)][pair_key] += (length_cm / genome_length_cm)
                break

    # Iterate through the rest of the trees
    for tree in tree_iter:
        current_left = tree.interval.left
        
        for (u, v) in pairs:
            new_mrca = tree.mrca(u, v)
            old_mrca, start_pos = current_state[(u, v)]
            
            # MRCA changed? Segment ended.
            if new_mrca != old_mrca:
                seg_len = current_left - start_pos
                process_segment(u, v, seg_len)
                
                # Start new segment
                current_state[(u, v)] = (new_mrca, current_left)

    # 4. Flush the final segments (end of genome)
    final_pos = ts.sequence_length
    for (u, v) in pairs:
        old_mrca, start_pos = current_state[(u, v)]
        seg_len = final_pos - start_pos
        process_segment(u, v, seg_len)

    # 5. Matrix Construction & Bootstrapping (Same as your original code)
    final_mean_matrix = {}
    final_var_matrix = {}

    for b_i in results:
        mean_matrix = np.zeros((num_pops, num_pops))
        var_matrix = np.zeros((num_pops, num_pops))

        for i in range(num_pops):
            for j in range(i, num_pops):
                # Calculate total theoretical pairs
                n_i = len(pop_samples[i])
                n_j = len(pop_samples[j])
                
                if i == j:
                    num_pairs = n_i * (n_i - 1) // 2
                else:
                    num_pairs = n_i * n_j
                
                if num_pairs == 0:
                    continue

                # Retrieve observed non-zero fractions
                observed_dict = results[b_i].get((i, j), {})
                observed_values = np.array(list(observed_dict.values()))
                
                # The rest are zeros
                count_zeros = num_pairs - len(observed_values)
                
                # Construct the full population of pairs
                full_population = np.concatenate([
                    observed_values, 
                    np.zeros(count_zeros)
                ])

                # A. Original Mean
                original_mean = np.mean(full_population)
                
                # B. Bootstrap Variance
                if num_bootstraps > 0:
                    boot_samples = np.random.choice(full_population, size=(num_bootstraps, num_pairs), replace=True)
                    boot_means = np.mean(boot_samples, axis=1)
                    boot_var = np.var(boot_means)
                else:
                    boot_var = 0.0

                # Fill Matrices
                mean_matrix[i, j] = mean_matrix[j, i] = original_mean
                var_matrix[i, j] = var_matrix[j, i] = boot_var

        final_mean_matrix[b_i] = mean_matrix
        final_var_matrix[b_i] = var_matrix

    return final_mean_matrix, final_var_matrix

In [82]:
mean, var = calculate_ibd_fractions_mrca(mutated_ts, bins, 1e-5)

Scanning trees for MRCA segments (780 pairs)...


In [83]:
mean

{0: array([[0.00302153]]), 1: array([[0.01338419]])}

In [85]:
var[0] ** (1/2)

array([[0.00011315]])